In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import os

DATASET_PATH = "/content/Talking_Dataset_MCropped"
TRAIN_PATH   = DATASET_PATH + "/Train"
VAL_PATH     = DATASET_PATH + "/Val"
SAVE_PATH    = "/content/drive/MyDrive/talking_final_clean.pth"

BATCH_SIZE   = 32
EPOCHS       = 20
IMG_SIZE     = 224
LR           = 3e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])


train_dataset = datasets.ImageFolder(TRAIN_PATH, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_PATH,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))


class MouthAttention(nn.Module):
    def __init__(self, in_channels=1280, spatial_size=7):
        super().__init__()
        self.spatial_size = spatial_size
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 128, 1),
            nn.ReLU(),
            nn.Conv2d(128, 1, 1),
        )

    def forward(self, x):
        B = x.size(0)
        attn = self.conv(x)
        attn = F.softmax(attn.view(B, -1), dim=1).view(B,1,self.spatial_size,self.spatial_size)
        out = (x * attn).mean(dim=[2,3])
        return out


class TalkingModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
        self.features = backbone.features
        self.attn = MouthAttention(1280, 7)

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        feat = self.features(x)
        pooled = self.attn(feat)
        return self.classifier(pooled)


model = TalkingModel().to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


def train_one_epoch():
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)



def validate():
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)

            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return acc, all_preds, all_labels



best_acc = 0

for epoch in range(1, EPOCHS+1):
    loss = train_one_epoch()
    acc, preds, labels = validate()

    print(f"[{epoch}/{EPOCHS}] Loss: {loss:.4f}  Val Acc: {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), SAVE_PATH)
        print("Saved best model")


print("\nFinal Evaluation")

model.load_state_dict(torch.load(SAVE_PATH))
acc, preds, labels = validate()

print("Accuracy:", acc)
print(classification_report(labels, preds, target_names=train_dataset.classes))
print("Confusion Matrix:\n", confusion_matrix(labels, preds))

In [ ]:
SAVE_PATH    = "/content/drive/MyDrive/talking_final_clean.pth"

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH))
acc, preds, labels = validate()

print("Accuracy:", acc)
print(classification_report(labels, preds, target_names=train_dataset.classes))
print("Confusion Matrix:\n", confusion_matrix(labels, preds))